### **Installations, Imports and Configurations**

In [ ]:
# ============================================================
# CELL 1 — Repair and install the RTX 5090 / CUDA 12.8 stack
# ============================================================

import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


def base_version(package):
    value = installed_version(package)
    return value.split("+")[0] if value else None


# Official matched CUDA 12.8 PyTorch set
EXPECTED = {
    "torch": "2.11.0",
    "torchvision": "0.26.0",
    "torchaudio": "2.11.0",
}

current = {name: base_version(name) for name in EXPECTED}

print("Python executable:", sys.executable)
print("Current PyTorch packages:", current)

repair_torch = any(
    current[name] != expected
    for name, expected in EXPECTED.items()
)

if repair_torch:
    print("\nRepairing the PyTorch installation...")

    # Remove the incompatible binary packages first
    subprocess.run(
        [
            sys.executable, "-m", "pip", "uninstall", "-y",
            "torch", "torchvision", "torchaudio", "torchao"
        ],
        check=False,
    )

    # Install the officially matched CUDA 12.8 wheels
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install",
            "--upgrade",
            "--force-reinstall",
            "--no-cache-dir",
            "torch==2.11.0",
            "torchvision==0.26.0",
            "torchaudio==2.11.0",
            "--index-url",
            "https://download.pytorch.org/whl/cu128",
        ]
    )
else:
    print("\nThe matched PyTorch 2.11.0 CUDA 12.8 stack is already installed.")


# Install the remaining notebook packages only after PyTorch
packages = [
    "torchao==0.17.0",
    "transformers==5.5.0",
    "peft==0.19.1",
    "accelerate==1.14.0",
    "datasets==4.3.0",
    "trl==0.24.0",
    "sacrebleu==2.6.0",
    "huggingface_hub>=0.34.0",
    "openai-harmony>=0.0.4",
    "sentencepiece>=0.2.0",
    "safetensors",
    "kernels",
    "bitsandbytes",
    "protobuf",
    "psutil>=6.0.0",
    "pandas",
    "tqdm",
]

print("\nInstalling the remaining notebook dependencies...")

subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install",
        "--upgrade",
        "--no-cache-dir",
        *packages,
    ]
)

print("\nInstalled versions:")
for package in [
    "torch",
    "torchvision",
    "torchaudio",
    "torchao",
    "transformers",
    "peft",
    "accelerate",
    "datasets",
    "trl",
]:
    print(f"{package:15s}: {installed_version(package)}")

print(
    "\nInstallation completed successfully.\n"
    "IMPORTANT: Restart the notebook kernel now, then start again from Cell 2."
)

In [2]:
import torch
import torchaudio
import transformers
import kernels

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Mxfp4Config,
)

print("Torch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

print("Torchaudio:", torchaudio.__version__)
print("Transformers:", transformers.__version__)
print("Kernels:", kernels.__version__)
print("\nEnvironment imports passed successfully.")

Torch: 2.11.0+cu128
CUDA runtime: 12.8
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
Compute capability: (12, 0)
Torchaudio: 2.11.0+cu128
Transformers: 5.5.0
Kernels: 0.14.1

Environment imports passed successfully.


In [3]:
# ============================================================
# Cell 2 — Imports and immutable experiment configuration
# ============================================================

import os
import gc
import json
import math
import time
import random
import hashlib
import zipfile

from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import sacrebleu
import torch

from datasets import (
    load_dataset,
    get_dataset_config_names,
    get_dataset_split_names,
)

from huggingface_hub import HfApi
from tqdm.auto import tqdm
from IPython.display import display

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Mxfp4Config,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"
)

SEED = 5090

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# ------------------------------------------------------------
# Main folders
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/home/mabdallah/alexandriax_mt_14d"
)

EXPERIMENT_NAME = (
    "gpt_oss_20b_dequant_bf16_greedy_"
    "noshot_context3_train_dev"
)

EXPERIMENT_DIR = (
    PROJECT_DIR
    / "teacher_benchmarks"
    / EXPERIMENT_NAME
)

DATA_DIR = EXPERIMENT_DIR / "prepared_data"

MODEL_CACHE_DIR = (
    PROJECT_DIR / "models" / "hf"
)

OFFLOAD_DIR = (
    EXPERIMENT_DIR / "bf16_cpu_disk_offload"
)

for directory in [
    EXPERIMENT_DIR,
    DATA_DIR,
    MODEL_CACHE_DIR,
    OFFLOAD_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# ------------------------------------------------------------
# Model and dataset
# ------------------------------------------------------------

MODEL_ID = "openai/gpt-oss-20b"
DATASET_NAME = "UBC-NLP/alexandria"

MAX_CONTEXT_TURNS = 3
MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 256

SAVE_EVERY = 100

EXPECTED_DEV_TURNS = 12250
EXPECTED_DEV_COUNTRIES = 11

PRECISION_MODE = (
    "dequantized_mxfp4_to_bfloat16"
)

PROMPT_VERSION = (
    "gpt_oss_teacher_noshot_"
    "context3_metadata_v1"
)

DECODE_TAG = (
    "greedy_no_sampling_beam1"
)

# ------------------------------------------------------------
# Pin exact Hugging Face revisions
# ------------------------------------------------------------

hub = HfApi()

MODEL_REVISION = (
    hub.model_info(MODEL_ID).sha
)

DATASET_REVISION = (
    hub.dataset_info(DATASET_NAME).sha
)

print("Experiment folder:")
print(EXPERIMENT_DIR)

print("\nModel:", MODEL_ID)
print("Model revision:", MODEL_REVISION)

print("\nDataset:", DATASET_NAME)
print("Dataset revision:", DATASET_REVISION)

print("\nPrecision:", PRECISION_MODE)
print("Prompt:", PROMPT_VERSION)
print("Decoding:", DECODE_TAG)
print("Save every:", SAVE_EVERY)

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Experiment folder:
/home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev

Model: openai/gpt-oss-20b
Model revision: 6cee5e81ee83917806bbde320786a8fb61efebee

Dataset: UBC-NLP/alexandria
Dataset revision: f2aa0479fd8a6373d247a2c8b58f099a9c5e5111

Precision: dequantized_mxfp4_to_bfloat16
Prompt: gpt_oss_teacher_noshot_context3_metadata_v1
Decoding: greedy_no_sampling_beam1
Save every: 100


### GPT-OSS Setup

In [4]:
# ============================================================
# Cell 3 — Hardware preflight for native GPT-OSS MXFP4
#
# OpenAI default:
#   - keep the released model weights in native MXFP4
#   - use torch_dtype="auto"
#   - use device_map="auto"
#   - do not dequantize the complete model to BF16
#
# Expected hardware:
#   - approximately 16 GiB VRAM for GPT-OSS-20B
#   - Hopper or newer NVIDIA architecture
#   - RTX 50-series GPUs are supported
# ============================================================

import json
import shutil
from importlib import metadata

import psutil
import torch


# ------------------------------------------------------------
# Official-style GPT-OSS loading configuration
# ------------------------------------------------------------

PRECISION_MODE = "native_mxfp4"
MODEL_TORCH_DTYPE = "auto"
MODEL_DEVICE_MAP = "auto"

MINIMUM_FREE_VRAM_GIB = 16.0
RECOMMENDED_FREE_VRAM_GIB = 20.0
MINIMUM_FREE_DISK_GIB = 25.0


# ------------------------------------------------------------
# Utility helpers
# ------------------------------------------------------------

def gib(number_of_bytes):
    return number_of_bytes / (1024 ** 3)


def installed_version(package_name):
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return None


# ------------------------------------------------------------
# Verify CUDA
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Native MXFP4 inference requires "
        "a supported NVIDIA GPU."
    )

device_index = torch.cuda.current_device()

gpu = torch.cuda.get_device_properties(
    device_index
)

compute_capability = torch.cuda.get_device_capability(
    device_index
)

torch.cuda.empty_cache()

free_vram_bytes, total_vram_bytes = (
    torch.cuda.mem_get_info(device_index)
)

gpu_total_gib = gib(total_vram_bytes)
gpu_free_gib = gib(free_vram_bytes)
gpu_used_gib = gpu_total_gib - gpu_free_gib


# ------------------------------------------------------------
# Verify supported GPU architecture
#
# Hopper begins at compute capability 9.x.
# RTX 50-series Blackwell GPUs are newer and also pass.
# ------------------------------------------------------------

compute_major, compute_minor = (
    compute_capability
)

if compute_major < 9:
    raise RuntimeError(
        "This GPU architecture is not supported for native "
        "MXFP4 execution by the official GPT-OSS Transformers "
        "setup.\n"
        f"Detected GPU: {gpu.name}\n"
        f"Compute capability: "
        f"{compute_major}.{compute_minor}\n"
        "Use a Hopper-or-newer GPU, including an RTX 50-series GPU."
    )


# ------------------------------------------------------------
# Verify model runtime dependencies
# ------------------------------------------------------------

package_versions = {
    "torch": installed_version("torch"),
    "transformers": installed_version(
        "transformers"
    ),
    "accelerate": installed_version(
        "accelerate"
    ),
    "triton": installed_version("triton"),
    "kernels": installed_version("kernels"),
    "openai-harmony": installed_version(
        "openai-harmony"
    ),
}

missing_packages = [
    package_name
    for package_name in [
        "transformers",
        "accelerate",
        "triton",
        "kernels",
        "openai-harmony",
    ]
    if package_versions[package_name] is None
]

if missing_packages:
    raise RuntimeError(
        "Missing packages required by the GPT-OSS native "
        "MXFP4 pipeline: "
        f"{missing_packages}\n"
        "Run the installation cell, restart the Python kernel, "
        "and then rerun this cell."
    )


# ------------------------------------------------------------
# Check VRAM
# ------------------------------------------------------------

if gpu_total_gib < MINIMUM_FREE_VRAM_GIB:
    raise RuntimeError(
        "GPT-OSS-20B requires approximately 16 GiB of GPU "
        "memory when using native MXFP4.\n"
        f"Detected total VRAM: {gpu_total_gib:.2f} GiB"
    )

if gpu_free_gib < MINIMUM_FREE_VRAM_GIB:
    raise RuntimeError(
        "There is not enough currently free GPU memory to load "
        "GPT-OSS-20B in native MXFP4.\n"
        f"Free VRAM: {gpu_free_gib:.2f} GiB\n"
        f"Required: approximately "
        f"{MINIMUM_FREE_VRAM_GIB:.0f} GiB\n"
        "Stop other GPU processes and rerun this cell."
    )


# ------------------------------------------------------------
# Check system RAM and model-cache disk
# ------------------------------------------------------------

ram = psutil.virtual_memory()

total_ram_gib = gib(ram.total)
available_ram_gib = gib(ram.available)

MODEL_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

disk_usage = shutil.disk_usage(
    MODEL_CACHE_DIR
)

free_disk_gib = gib(disk_usage.free)

if free_disk_gib < MINIMUM_FREE_DISK_GIB:
    raise RuntimeError(
        "There may not be enough free disk space to cache "
        "GPT-OSS-20B.\n"
        f"Free disk space: {free_disk_gib:.2f} GiB\n"
        f"Required safety minimum: "
        f"{MINIMUM_FREE_DISK_GIB:.0f} GiB"
    )


# ------------------------------------------------------------
# Display preflight information
# ------------------------------------------------------------

print("=" * 72)
print("GPT-OSS-20B NATIVE MXFP4 PREFLIGHT")
print("=" * 72)

print(f"GPU: {gpu.name}")
print(
    "Compute capability:",
    f"{compute_major}.{compute_minor}",
)
print(f"CUDA runtime: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

print(
    f"\nGPU VRAM total: {gpu_total_gib:.2f} GiB"
)
print(
    f"GPU VRAM free:  {gpu_free_gib:.2f} GiB"
)
print(
    f"GPU VRAM used:  {gpu_used_gib:.2f} GiB"
)

print(
    f"\nSystem RAM total:     "
    f"{total_ram_gib:.2f} GiB"
)
print(
    f"System RAM available: "
    f"{available_ram_gib:.2f} GiB"
)
print(
    f"Model-cache disk free: "
    f"{free_disk_gib:.2f} GiB"
)

print("\nModel loading configuration:")
print(f"  Precision:  {PRECISION_MODE}")
print(f"  dtype:      {MODEL_TORCH_DTYPE}")
print(f"  device map: {MODEL_DEVICE_MAP}")

print("\nInstalled packages:")

for package_name, package_version in (
    package_versions.items()
):
    print(
        f"  {package_name}: "
        f"{package_version}"
    )

if gpu_free_gib < RECOMMENDED_FREE_VRAM_GIB:
    print(
        "\nWARNING: The model should fit, but less than "
        f"{RECOMMENDED_FREE_VRAM_GIB:.0f} GiB VRAM is free. "
        "Long inputs may leave limited room for the KV cache."
    )


# ------------------------------------------------------------
# Save reproducibility record
# ------------------------------------------------------------

preflight = {
    "experiment_name": EXPERIMENT_NAME,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_name": DATASET_NAME,
    "dataset_revision": DATASET_REVISION,
    "precision_mode": PRECISION_MODE,
    "model_torch_dtype": MODEL_TORCH_DTYPE,
    "model_device_map": MODEL_DEVICE_MAP,
    "gpu": gpu.name,
    "compute_capability": (
        f"{compute_major}.{compute_minor}"
    ),
    "cuda_runtime": torch.version.cuda,
    "gpu_total_gib": round(
        gpu_total_gib,
        4,
    ),
    "gpu_free_gib": round(
        gpu_free_gib,
        4,
    ),
    "total_ram_gib": round(
        total_ram_gib,
        4,
    ),
    "available_ram_gib": round(
        available_ram_gib,
        4,
    ),
    "free_disk_gib": round(
        free_disk_gib,
        4,
    ),
    "package_versions": package_versions,
    "save_every": SAVE_EVERY,
}

with open(
    EXPERIMENT_DIR
    / "experiment_preflight.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preflight,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "\n✅ Hardware preflight passed for native MXFP4."
)

GPT-OSS-20B NATIVE MXFP4 PREFLIGHT
GPU: NVIDIA GeForce RTX 5090
Compute capability: 12.0
CUDA runtime: 12.8
PyTorch: 2.11.0+cu128

GPU VRAM total: 31.36 GiB
GPU VRAM free:  30.26 GiB
GPU VRAM used:  1.10 GiB

System RAM total:     31.06 GiB
System RAM available: 25.28 GiB
Model-cache disk free: 500.00 GiB

Model loading configuration:
  Precision:  native_mxfp4
  dtype:      auto
  device map: auto

Installed packages:
  torch: 2.11.0+cu128
  transformers: 5.5.0
  accelerate: 1.14.0
  triton: 3.6.0
  kernels: 0.14.1
  openai-harmony: 0.0.8

✅ Hardware preflight passed for native MXFP4.


### **Data Setup**

In [5]:
# ============================================================
# Cell 4 — Prepare complete TRAIN and official DEV data
#
# Preserved from the Alexandria notebooks:
#   - country/config
#   - dialect and domain
#   - participants
#   - speaker
#   - gender direction
#   - previous three English turns
#   - current English turn
#   - gold dialectal-Arabic reference
# ============================================================

def to_plain(value):
    if isinstance(value, np.ndarray):
        return [
            to_plain(x)
            for x in value.tolist()
        ]

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, (tuple, list)):
        return [
            to_plain(x)
            for x in value
        ]

    if isinstance(value, dict):
        return {
            str(key): to_plain(item)
            for key, item in value.items()
        }

    return value


def scalar_text(value):
    value = to_plain(value)

    if value is None:
        return ""

    if isinstance(value, (list, dict)):
        return str(value).strip()

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


def normalize_turn_list(value):
    value = to_plain(value)

    if value is None:
        return []

    if isinstance(value, list):
        normalized = []

        for item in value:
            if item is None:
                continue

            if isinstance(item, dict):
                normalized.append(item)
            else:
                normalized.append({
                    "text": scalar_text(item)
                })

        return normalized

    if isinstance(value, dict):
        lengths = [
            len(item)
            for item in value.values()
            if isinstance(item, list)
        ]

        if not lengths:
            return []

        rows = []

        for index in range(max(lengths)):
            one_row = {}

            for key, items in value.items():
                if isinstance(items, list):
                    one_row[key] = (
                        items[index]
                        if index < len(items)
                        else None
                    )
                else:
                    one_row[key] = items

            rows.append(one_row)

        return rows

    return []


def turn_field(
    turn,
    keys,
    default="",
):
    turn = to_plain(turn)

    if not isinstance(turn, dict):
        return default

    for key in keys:
        if key not in turn:
            continue

        text = scalar_text(turn[key])

        if text:
            return text

    return default


def turn_text(turn):
    return turn_field(
        turn,
        [
            "text",
            "sentence",
            "utterance",
            "content",
            "value",
            "english",
            "source_text",
            "target_arabic",
            "translation",
        ],
        default="",
    )


def extract_turn_order(
    turn,
    fallback_index,
):
    raw_order = turn_field(
        turn,
        [
            "turn_order",
            "turn_id",
            "order",
            "idx",
            "index",
        ],
        default="",
    )

    if raw_order:
        try:
            return int(raw_order)
        except Exception:
            pass

    return int(fallback_index + 1)


def flatten_alexandria_split(
    dataset,
    config_name,
    split_name,
):
    records = []

    for conversation_index, raw_row in enumerate(
        dataset
    ):
        row = to_plain(dict(raw_row))

        conversation_id = scalar_text(
            row.get(
                "conv_id",
                row.get(
                    "conversation_id",
                    (
                        f"{config_name}_"
                        f"{split_name}_"
                        f"{conversation_index}"
                    ),
                ),
            )
        )

        country = (
            scalar_text(
                row.get(
                    "country",
                    config_name,
                )
            )
            or config_name
        )

        dialect = scalar_text(
            row.get("dialect", "")
        )

        domain = scalar_text(
            row.get("domain", "")
        )

        participants = scalar_text(
            row.get("participants", "")
        )

        english_turns = normalize_turn_list(
            row.get(
                "english_conversation",
                [],
            )
        )

        arabic_turns = normalize_turn_list(
            row.get(
                "dialectal_conversation",
                [],
            )
        )

        number_of_turns = min(
            len(english_turns),
            len(arabic_turns),
        )

        for turn_index in range(
            number_of_turns
        ):
            english_turn = (
                english_turns[turn_index]
            )

            arabic_turn = (
                arabic_turns[turn_index]
            )

            source_text = turn_text(
                english_turn
            )

            reference_arabic = turn_text(
                arabic_turn
            )

            if (
                not source_text
                or not reference_arabic
            ):
                continue

            english_order = (
                extract_turn_order(
                    english_turn,
                    turn_index,
                )
            )

            arabic_order = (
                extract_turn_order(
                    arabic_turn,
                    turn_index,
                )
            )

            if english_order != arabic_order:
                raise RuntimeError(
                    "English/Arabic turn mismatch: "
                    f"{config_name}/"
                    f"{split_name}/"
                    f"{conversation_id}: "
                    f"{english_order} != "
                    f"{arabic_order}"
                )

            previous_english_turns = []

            previous_start = max(
                0,
                turn_index
                - MAX_CONTEXT_TURNS,
            )

            for previous_index in range(
                previous_start,
                turn_index,
            ):
                previous_turn = (
                    english_turns[
                        previous_index
                    ]
                )

                previous_english_turns.append({
                    "turn_order": (
                        extract_turn_order(
                            previous_turn,
                            previous_index,
                        )
                    ),
                    "speaker": turn_field(
                        previous_turn,
                        [
                            "speaker",
                            "role",
                            "speaker_role",
                            "participant",
                        ],
                    ),
                    "direction": turn_field(
                        previous_turn,
                        [
                            "direction",
                            "gender_direction",
                            (
                                "speaker_"
                                "addressee_gender"
                            ),
                        ],
                    ),
                    "text": turn_text(
                        previous_turn
                    ),
                })

            speaker = turn_field(
                english_turn,
                [
                    "speaker",
                    "role",
                    "speaker_role",
                    "participant",
                ],
            )

            gender_direction = turn_field(
                english_turn,
                [
                    "direction",
                    "gender_direction",
                    (
                        "speaker_"
                        "addressee_gender"
                    ),
                ],
            )

            source_id = (
                f"{config_name}_"
                f"{split_name}_"
                f"{conversation_id}_"
                f"{english_order}"
            )

            records.append({
                "source_id": source_id,
                "config": config_name,
                "country": country,
                "split": split_name,
                "conversation_id": (
                    conversation_id
                ),
                "turn_order": int(
                    english_order
                ),
                "dialect": dialect,
                "domain": domain,
                "participants": participants,
                "speaker": speaker,
                "gender_direction": (
                    gender_direction
                ),
                "previous_english_turns": (
                    previous_english_turns
                ),
                "source_text": source_text,
                "reference_arabic": (
                    reference_arabic
                ),
            })

    return records


def dataframe_hash(frame):
    digest = hashlib.sha256()

    for row in frame[
        [
            "source_id",
            "source_text",
            "reference_arabic",
        ]
    ].itertuples(index=False):

        digest.update(
            "\x1f".join(
                map(str, row)
            ).encode("utf-8")
        )

        digest.update(b"\n")

    return digest.hexdigest()


TRAIN_CACHE_PATH = (
    DATA_DIR
    / "complete_train_turns.pkl"
)

DEV_CACHE_PATH = (
    DATA_DIR
    / "official_dev_turns.pkl"
)

DATA_REPORT_PATH = (
    DATA_DIR
    / "prepared_data_report.json"
)

cache_valid = False

if (
    TRAIN_CACHE_PATH.exists()
    and DEV_CACHE_PATH.exists()
    and DATA_REPORT_PATH.exists()
):
    with open(
        DATA_REPORT_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        old_report = json.load(f)

    cache_valid = (
        old_report.get(
            "dataset_revision"
        )
        == DATASET_REVISION
        and old_report.get(
            "max_context_turns"
        )
        == MAX_CONTEXT_TURNS
    )

if cache_valid:
    train_df = pd.read_pickle(
        TRAIN_CACHE_PATH
    )

    dev_df = pd.read_pickle(
        DEV_CACHE_PATH
    )

    print(
        "Loaded prepared TRAIN/DEV cache."
    )

else:
    available_configs = (
        get_dataset_config_names(
            DATASET_NAME,
            revision=DATASET_REVISION,
        )
    )

    split_map = {
        config: get_dataset_split_names(
            DATASET_NAME,
            config,
            revision=DATASET_REVISION,
        )
        for config in available_configs
    }

    train_configs = sorted([
        config
        for config in available_configs
        if "train" in split_map[config]
    ])

    dev_configs = sorted([
        config
        for config in available_configs
        if "dev" in split_map[config]
    ])

    print(
        "TRAIN configurations:",
        train_configs,
    )

    print(
        "DEV configurations:",
        dev_configs,
    )

    train_records = []
    dev_records = []

    for config in train_configs:
        print(
            f"Loading {config}/train ..."
        )

        dataset = load_dataset(
            DATASET_NAME,
            config,
            split="train",
            revision=DATASET_REVISION,
        )

        train_records.extend(
            flatten_alexandria_split(
                dataset,
                config,
                "train",
            )
        )

    for config in dev_configs:
        print(
            f"Loading {config}/dev ..."
        )

        dataset = load_dataset(
            DATASET_NAME,
            config,
            split="dev",
            revision=DATASET_REVISION,
        )

        dev_records.extend(
            flatten_alexandria_split(
                dataset,
                config,
                "dev",
            )
        )

    train_df = (
        pd.DataFrame(train_records)
        .sort_values([
            "config",
            "conversation_id",
            "turn_order",
        ])
        .reset_index(drop=True)
    )

    dev_df = (
        pd.DataFrame(dev_records)
        .sort_values([
            "config",
            "conversation_id",
            "turn_order",
        ])
        .reset_index(drop=True)
    )

    train_df.to_pickle(
        TRAIN_CACHE_PATH
    )

    dev_df.to_pickle(
        DEV_CACHE_PATH
    )

for split_name, frame in [
    ("train", train_df),
    ("dev", dev_df),
]:
    frame["source_id"] = (
        frame["source_id"].astype(str)
    )

    frame["config"] = (
        frame["config"].astype(str)
    )

    frame["conversation_id"] = (
        frame[
            "conversation_id"
        ].astype(str)
    )

    frame["turn_order"] = (
        pd.to_numeric(
            frame["turn_order"],
            errors="raise",
        ).astype(int)
    )

    frame["source_text"] = (
        frame["source_text"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    frame["reference_arabic"] = (
        frame["reference_arabic"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    if frame[
        "source_id"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate {split_name} "
            "source IDs."
        )

    if (
        frame["source_text"] == ""
    ).any():
        raise RuntimeError(
            f"Empty {split_name} "
            "source text."
        )

    if (
        frame["reference_arabic"]
        == ""
    ).any():
        raise RuntimeError(
            f"Empty {split_name} "
            "gold reference."
        )

if len(dev_df) != EXPECTED_DEV_TURNS:
    raise RuntimeError(
        f"Expected {EXPECTED_DEV_TURNS} "
        f"DEV turns; found {len(dev_df)}."
    )

if (
    dev_df["config"].nunique()
    != EXPECTED_DEV_COUNTRIES
):
    raise RuntimeError(
        f"Expected "
        f"{EXPECTED_DEV_COUNTRIES} "
        "DEV countries; found "
        f"{dev_df['config'].nunique()}."
    )

data_report = {
    "dataset_name": DATASET_NAME,
    "dataset_revision": (
        DATASET_REVISION
    ),
    "max_context_turns": (
        MAX_CONTEXT_TURNS
    ),
    "train_turns": int(
        len(train_df)
    ),
    "dev_turns": int(
        len(dev_df)
    ),
    "train_configs": sorted(
        train_df[
            "config"
        ].unique().tolist()
    ),
    "dev_configs": sorted(
        dev_df[
            "config"
        ].unique().tolist()
    ),
    "train_hash": dataframe_hash(
        train_df
    ),
    "dev_hash": dataframe_hash(
        dev_df
    ),
}

with open(
    DATA_REPORT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        data_report,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    "\nTRAIN turns:",
    len(train_df),
)

print(
    "DEV turns:",
    len(dev_df),
)

display(
    pd.concat(
        [
            train_df.groupby(
                "config"
            ).size().rename(
                "train_turns"
            ),
            dev_df.groupby(
                "config"
            ).size().rename(
                "dev_turns"
            ),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

print(
    "\n✅ Complete TRAIN and "
    "official DEV data are ready."
)

TRAIN configurations: ['EG', 'JO', 'LB', 'MA', 'MR', 'OM', 'PS', 'SA', 'SY', 'TN', 'YE']
DEV configurations: ['EG', 'JO', 'LB', 'MA', 'MR', 'OM', 'PS', 'SA', 'SY', 'TN', 'YE']
Loading EG/train ...
Loading JO/train ...
Loading LB/train ...
Loading MA/train ...
Loading MR/train ...
Loading OM/train ...
Loading PS/train ...
Loading SA/train ...
Loading SY/train ...
Loading TN/train ...
Loading YE/train ...
Loading EG/dev ...
Loading JO/dev ...
Loading LB/dev ...
Loading MA/dev ...
Loading MR/dev ...
Loading OM/dev ...
Loading PS/dev ...
Loading SA/dev ...
Loading SY/dev ...
Loading TN/dev ...
Loading YE/dev ...

TRAIN turns: 66480
DEV turns: 12250


,config,train_turns,dev_turns
0,EG,3108,1113
1,JO,5501,1113
2,LB,8906,1118
3,MA,2573,1110
4,MR,5515,1114
5,OM,6280,1109
6,PS,14933,1110
7,SA,8470,1110
8,SY,6071,1119
9,TN,2034,1116



✅ Complete TRAIN and official DEV data are ready.


### **Load the GPT-OSS**

In [7]:
# ============================================================
# Cell 5 — Load GPT-OSS-20B in native MXFP4
#
# OpenAI-standard loading:
#   - native released MXFP4 weights
#   - torch_dtype="auto"
#   - device_map="auto"
#   - no Mxfp4Config(dequantize=True)
#   - no BF16 model expansion
#   - no CPU/disk weight offloading
# ============================================================

import gc
import json

import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

from openai_harmony import (
    HarmonyEncodingName,
    Role,
    load_harmony_encoding,
)


# ------------------------------------------------------------
# Release unused CUDA allocations before model loading
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

print("Loading GPT-OSS tokenizer...")

teacher_tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=str(MODEL_CACHE_DIR),
        use_fast=True,
    )
)

teacher_tokenizer.padding_side = "left"
teacher_tokenizer.truncation_side = "left"

if teacher_tokenizer.pad_token_id is None:
    if teacher_tokenizer.eos_token_id is None:
        raise RuntimeError(
            "The tokenizer has neither a padding token "
            "nor an EOS token."
        )

    teacher_tokenizer.pad_token = (
        teacher_tokenizer.eos_token
    )


# ------------------------------------------------------------
# Load native MXFP4 model
# ------------------------------------------------------------

print(
    "Loading GPT-OSS-20B using native MXFP4 "
    "with torch_dtype='auto'..."
)

teacher_model = (
    AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=str(MODEL_CACHE_DIR),

        # Official OpenAI/Hugging Face loading behavior.
        torch_dtype=MODEL_TORCH_DTYPE,
        device_map=MODEL_DEVICE_MAP,
    )
)

teacher_model.eval()
teacher_model.config.use_cache = True

teacher_model.generation_config.do_sample = False
teacher_model.generation_config.num_beams = 1

# Avoid decoding warnings when sampling is disabled.
teacher_model.generation_config.temperature = None
teacher_model.generation_config.top_p = None
teacher_model.generation_config.top_k = None


# ------------------------------------------------------------
# Verify native MXFP4 configuration
# ------------------------------------------------------------

quantization_config = getattr(
    teacher_model.config,
    "quantization_config",
    None,
)

if hasattr(quantization_config, "to_dict"):
    quantization_record = (
        quantization_config.to_dict()
    )
else:
    quantization_record = quantization_config

quantization_description = json.dumps(
    quantization_record,
    ensure_ascii=False,
    default=str,
    sort_keys=True,
)

if "mxfp4" not in quantization_description.lower():
    raise RuntimeError(
        "The loaded model configuration does not report "
        "native MXFP4 quantization.\n"
        f"Quantization configuration: "
        f"{quantization_description}"
    )


# ------------------------------------------------------------
# Ensure the model was not offloaded
# ------------------------------------------------------------

device_map = getattr(
    teacher_model,
    "hf_device_map",
    {},
)

offloaded_modules = {
    module_name: placement
    for module_name, placement in device_map.items()
    if str(placement).lower() in {
        "cpu",
        "disk",
    }
}

if offloaded_modules:
    raise RuntimeError(
        "Some model modules were offloaded outside the GPU. "
        "This should not be necessary for GPT-OSS-20B in "
        "native MXFP4 on the RTX 5090.\n"
        f"First offloaded modules: "
        f"{list(offloaded_modules.items())[:10]}"
    )


# ------------------------------------------------------------
# Initialize official Harmony encoding
# ------------------------------------------------------------

harmony = load_harmony_encoding(
    HarmonyEncodingName.HARMONY_GPT_OSS
)

HARMONY_STOP_TOKEN_IDS = sorted(
    int(token_id)
    for token_id in (
        harmony
        .stop_tokens_for_assistant_actions()
    )
)

HARMONY_STOP_TOKEN_ID_SET = set(
    HARMONY_STOP_TOKEN_IDS
)


# ------------------------------------------------------------
# Identify the device where token inputs must be placed
# ------------------------------------------------------------

INPUT_DEVICE = (
    teacher_model
    .get_input_embeddings()
    .weight
    .device
)

if INPUT_DEVICE.type != "cuda":
    raise RuntimeError(
        "The model input embeddings are not on CUDA.\n"
        f"Detected input device: {INPUT_DEVICE}"
    )


# ------------------------------------------------------------
# Runtime report
# ------------------------------------------------------------

parameter_dtypes = sorted({
    str(parameter.dtype)
    for parameter in teacher_model.parameters()
})

model_footprint_gib = (
    teacher_model.get_memory_footprint()
    / (1024 ** 3)
)

allocated_gib = (
    torch.cuda.memory_allocated()
    / (1024 ** 3)
)

reserved_gib = (
    torch.cuda.memory_reserved()
    / (1024 ** 3)
)

print("\nModel loading report")
print("-" * 72)
print("Model:", MODEL_ID)
print("Precision:", PRECISION_MODE)
print("torch_dtype:", MODEL_TORCH_DTYPE)
print("Input device:", INPUT_DEVICE)
print("Parameter dtypes:", parameter_dtypes)
print("Quantization:", quantization_record)
print("Device map:", device_map)

print(
    "Reported model footprint:",
    f"{model_footprint_gib:.2f} GiB",
)

print(
    "CUDA memory allocated:",
    f"{allocated_gib:.2f} GiB",
)

print(
    "CUDA memory reserved:",
    f"{reserved_gib:.2f} GiB",
)

print(
    "Harmony stop token IDs:",
    HARMONY_STOP_TOKEN_IDS,
)

print(
    "\n✅ GPT-OSS-20B loaded in native MXFP4."
)

Loading GPT-OSS tokenizer...


config.json:   0%|          | 0.00/1.81k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.20k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 27.9MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 43 files:   0%|          | 0/43 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]


Model loading report
------------------------------------------------------------------------
Model: openai/gpt-oss-20b
Precision: native_mxfp4
torch_dtype: auto
Input device: cuda:0
Parameter dtypes: ['torch.bfloat16', 'torch.float32']
Quantization: {'quant_method': <QuantizationMethod.MXFP4: 'mxfp4'>, 'modules_to_not_convert': ['model.layers.*.self_attn', 'model.layers.*.mlp.router', 'model.embed_tokens', 'lm_head']}
Device map: {}
Reported model footprint: 3.37 GiB
CUDA memory allocated: 12.83 GiB
CUDA memory reserved: 12.85 GiB
Harmony stop token IDs: [200002, 200012]

✅ GPT-OSS-20B loaded in native MXFP4.


### Prompting, generation, scoring and resume engine

In [8]:
# ============================================================
# Cell 6 — GPT-OSS teacher benchmark engine
#
# Features:
#   - native MXFP4 inference
#   - official Transformers chat template
#   - Harmony message parsing
#   - low reasoning effort
#   - deterministic greedy decoding
#   - context and metadata without demonstrations
#   - atomic saving every SAVE_EVERY predictions
#   - safe resume after interruption
#   - official-style per-country metrics
#   - predictions.jsonl and submission ZIP
# ============================================================

import hashlib
import json
import os
import time
import zipfile

import pandas as pd
import sacrebleu
import torch

from IPython.display import display
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Prompt and decoding configuration
# ------------------------------------------------------------

REASONING_EFFORT = "low"

SYSTEM_PROMPT = """
You are an expert context-aware machine translation system.

Translate the current English dialogue turn into natural Arabic
in the requested country's local dialect.

Use the metadata and previous English dialogue only as context.
Preserve the meaning, names, numbers, entities, tone, gender,
speaker relationships, and level of formality.

Do not explain or discuss the translation.
Put only the Arabic translation in the final response.
""".strip()


# ------------------------------------------------------------
# General text helpers
# ------------------------------------------------------------

def safe_text(value):
    if value is None:
        return ""

    if isinstance(value, str):
        return value.strip()

    try:
        missing = pd.isna(value)

        if isinstance(missing, bool) and missing:
            return ""
    except Exception:
        pass

    return str(value).strip()


def clean_translation(text):
    text = (
        safe_text(text)
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .strip()
        .strip("`")
        .strip()
    )

    prefixes = [
        "Arabic translation:",
        "Arabic Translation:",
        "Translation:",
        "Arabic:",
        "الترجمة العربية:",
        "الترجمة:",
    ]

    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[
                len(prefix):
            ].strip()
            break

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    return (
        " ".join(lines)
        .strip()
        .strip("\"'")
        .strip()
    )


def arabic_ratio(text):
    visible_characters = [
        character
        for character in safe_text(text)
        if not character.isspace()
    ]

    if not visible_characters:
        return 0.0

    arabic_characters = sum(
        (
            "\u0600" <= character <= "\u06ff"
        )
        or (
            "\u0750" <= character <= "\u077f"
        )
        or (
            "\u08a0" <= character <= "\u08ff"
        )
        or (
            "\ufb50" <= character <= "\ufdff"
        )
        or (
            "\ufe70" <= character <= "\ufeff"
        )
        for character in visible_characters
    )

    return (
        arabic_characters
        / len(visible_characters)
    )


# ------------------------------------------------------------
# Build dialogue context
# ------------------------------------------------------------

def build_context_block(row):
    previous_turns = to_plain(
        row.get(
            "previous_english_turns",
            [],
        )
    )

    if not previous_turns:
        return "No previous context."

    lines = []

    for context_index, previous_turn in enumerate(
        previous_turns,
        start=1,
    ):
        if isinstance(previous_turn, dict):
            text = safe_text(
                previous_turn.get(
                    "text",
                    "",
                )
            )

            speaker = safe_text(
                previous_turn.get(
                    "speaker",
                    "",
                )
            )

            direction = safe_text(
                previous_turn.get(
                    "direction",
                    "",
                )
            )
        else:
            text = safe_text(previous_turn)
            speaker = ""
            direction = ""

        if not text:
            continue

        speaker_metadata = " / ".join(
            item
            for item in [
                speaker,
                direction,
            ]
            if item
        )

        if speaker_metadata:
            lines.append(
                f"{context_index}. "
                f"[{speaker_metadata}] "
                f"{text}"
            )
        else:
            lines.append(
                f"{context_index}. {text}"
            )

    if not lines:
        return "No previous context."

    return "\n".join(lines)


def build_messages(row):
    metadata_fields = [
        (
            "Country/config",
            row.get("config", ""),
        ),
        (
            "Country",
            row.get("country", ""),
        ),
        (
            "Target dialect",
            row.get("dialect", ""),
        ),
        (
            "Domain",
            row.get("domain", ""),
        ),
        (
            "Participants/persona",
            row.get("participants", ""),
        ),
        (
            "Current speaker",
            row.get("speaker", ""),
        ),
        (
            "Speaker-to-addressee gender direction",
            row.get("gender_direction", ""),
        ),
    ]

    metadata_lines = []

    for label, value in metadata_fields:
        value = safe_text(value)

        if value:
            metadata_lines.append(
                f"{label}: {value}"
            )

    metadata_text = (
        "\n".join(metadata_lines)
        if metadata_lines
        else "No metadata."
    )

    user_prompt = f"""
Translate the current English dialogue turn into the requested
local Arabic dialect.

Metadata:
{metadata_text}

Previous English dialogue context:
{build_context_block(row)}

Current English turn:
{safe_text(row.get("source_text", ""))}

Return only the Arabic translation in the final response.
""".strip()

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


# ------------------------------------------------------------
# Harmony output parsing
# ------------------------------------------------------------

def harmony_content_to_text(content):
    if content is None:
        return ""

    if isinstance(content, str):
        return content

    if hasattr(content, "to_dict"):
        content = content.to_dict()

    if isinstance(content, list):
        parts = [
            harmony_content_to_text(item)
            for item in content
        ]

        return "\n".join(
            part
            for part in parts
            if part
        )

    if isinstance(content, dict):
        for key in [
            "text",
            "content",
            "value",
        ]:
            if key in content:
                extracted = harmony_content_to_text(
                    content[key]
                )

                if extracted:
                    return extracted

        return ""

    return safe_text(content)


def extract_final_channel(
    completion_ids,
    raw_completion,
):
    parsed_records = []
    final_candidates = []
    parse_error = ""

    try:
        parsed_messages = (
            harmony
            .parse_messages_from_completion_tokens(
                completion_ids,
                Role.ASSISTANT,
            )
        )

        for message in parsed_messages:
            if hasattr(message, "to_dict"):
                message_record = message.to_dict()
            elif isinstance(message, dict):
                message_record = message
            else:
                message_record = {
                    "raw": safe_text(message)
                }

            parsed_records.append(
                message_record
            )

            channel = safe_text(
                message_record.get(
                    "channel",
                    "",
                )
            ).lower()

            normalized_channel = (
                channel.split(".")[-1]
            )

            if normalized_channel != "final":
                continue

            final_text = harmony_content_to_text(
                message_record.get(
                    "content",
                    "",
                )
            )

            final_text = clean_translation(
                final_text
            )

            if final_text:
                final_candidates.append(
                    final_text
                )

    except Exception as error:
        parse_error = (
            f"{type(error).__name__}: "
            f"{str(error)[:1000]}"
        )

    # Primary path: official Harmony parser.
    if final_candidates:
        return {
            "prediction": final_candidates[-1],
            "parsed_messages": parsed_records,
            "harmony_parse_error": parse_error,
            "extraction_method": (
                "openai_harmony_parser"
            ),
        }

    # Defensive fallback for debugging incomplete generations.
    marker = (
        "<|channel|>"
        "final"
        "<|message|>"
    )

    if marker in raw_completion:
        final_text = (
            raw_completion
            .rsplit(marker, 1)[-1]
        )

        for end_marker in [
            "<|return|>",
            "<|end|>",
            "<|start|>",
            "<|call|>",
        ]:
            final_text = (
                final_text
                .split(end_marker, 1)[0]
            )

        final_text = clean_translation(
            final_text
        )

        if final_text:
            return {
                "prediction": final_text,
                "parsed_messages": parsed_records,
                "harmony_parse_error": parse_error,
                "extraction_method": (
                    "decoded_final_marker_fallback"
                ),
            }

    return {
        "prediction": "",
        "parsed_messages": parsed_records,
        "harmony_parse_error": parse_error,
        "extraction_method": "failed",
    }


# ------------------------------------------------------------
# Generate one translation
# ------------------------------------------------------------

def generate_one(row):
    messages = build_messages(row)

    encoded = (
        teacher_tokenizer
        .apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,

            # Official GPT-OSS reasoning control.
            reasoning_effort=REASONING_EFFORT,
        )
    )

    encoded = {
        key: value.to(INPUT_DEVICE)
        for key, value in encoded.items()
    }

    prompt_tokens = int(
        encoded["input_ids"].shape[-1]
    )

    with torch.inference_mode():
        generated = teacher_model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,

            # Deterministic benchmark decoding.
            do_sample=False,
            num_beams=1,

            eos_token_id=(
                HARMONY_STOP_TOKEN_IDS
            ),

            pad_token_id=(
                teacher_tokenizer.pad_token_id
            ),

            use_cache=True,
        )

    completion_ids = (
        generated[
            0,
            prompt_tokens:,
        ]
        .detach()
        .cpu()
        .tolist()
    )

    raw_completion = (
        teacher_tokenizer.decode(
            completion_ids,
            skip_special_tokens=False,
        )
    )

    extraction = extract_final_channel(
        completion_ids=completion_ids,
        raw_completion=raw_completion,
    )

    prediction = extraction["prediction"]

    reached_token_limit = (
        len(completion_ids)
        >= MAX_NEW_TOKENS
        and (
            not completion_ids
            or completion_ids[-1]
            not in HARMONY_STOP_TOKEN_ID_SET
        )
    )

    generation_error = ""

    if reached_token_limit:
        generation_error = (
            "max_new_tokens_reached_"
            "before_harmony_stop"
        )

    elif not prediction:
        generation_error = (
            "missing_or_empty_"
            "harmony_final"
        )

    elif arabic_ratio(prediction) < 0.10:
        generation_error = (
            "final_output_has_"
            "low_arabic_ratio"
        )

    return {
        "raw_completion": raw_completion,
        "prediction": prediction,
        "generation_error": generation_error,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": len(
            completion_ids
        ),
        "harmony_message_count": len(
            extraction["parsed_messages"]
        ),
        "harmony_parse_error": (
            extraction[
                "harmony_parse_error"
            ]
        ),
        "extraction_method": (
            extraction[
                "extraction_method"
            ]
        ),
    }


# ------------------------------------------------------------
# Atomic file-writing helpers
# ------------------------------------------------------------

def atomic_csv_write(
    dataframe,
    destination,
):
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        destination.with_name(
            destination.name + ".tmp"
        )
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig",
    )

    os.replace(
        temporary_path,
        destination,
    )


def atomic_json_write(
    payload,
    destination,
):
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        destination.with_name(
            destination.name + ".tmp"
        )
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            ensure_ascii=False,
            indent=2,
        )

    os.replace(
        temporary_path,
        destination,
    )


# ------------------------------------------------------------
# Experiment fingerprint
# ------------------------------------------------------------

def experiment_fingerprint(
    frame,
    split_name,
):
    payload = {
        "split": split_name,
        "data_hash": dataframe_hash(frame),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "precision_mode": PRECISION_MODE,
        "model_torch_dtype": (
            MODEL_TORCH_DTYPE
        ),
        "model_device_map": (
            MODEL_DEVICE_MAP
        ),
        "reasoning_effort": (
            REASONING_EFFORT
        ),
        "prompt_version": PROMPT_VERSION,
        "decode_tag": DECODE_TAG,
        "decoding": "greedy",
        "num_beams": 1,
        "do_sample": False,
        "max_input_tokens": (
            MAX_INPUT_TOKENS
        ),
        "max_new_tokens": (
            MAX_NEW_TOKENS
        ),
        "context_turns": (
            MAX_CONTEXT_TURNS
        ),
        "harmony_parser": True,
    }

    fingerprint = hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()

    return fingerprint, payload


# ------------------------------------------------------------
# Score and package completed predictions
# ------------------------------------------------------------

def score_and_package(
    frame,
    prediction_df,
    split_name,
    split_dir,
    fingerprint,
):
    expected_ids = set(
        frame["source_id"].astype(str)
    )

    actual_ids = set(
        prediction_df[
            "source_id"
        ].astype(str)
    )

    if expected_ids != actual_ids:
        raise RuntimeError(
            f"{split_name}: prediction ID mismatch. "
            f"Missing="
            f"{len(expected_ids - actual_ids)}, "
            f"extra="
            f"{len(actual_ids - expected_ids)}"
        )

    if prediction_df[
        "source_id"
    ].duplicated().any():
        raise RuntimeError(
            f"{split_name}: duplicate prediction IDs."
        )

    prediction_errors = (
        prediction_df[
            "generation_error"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    if prediction_errors.ne("").any():
        raise RuntimeError(
            f"{split_name}: predictions still "
            "contain generation errors."
        )

    scored_df = frame[
        [
            "source_id",
            "config",
            "country",
            "conversation_id",
            "turn_order",
            "source_text",
            "reference_arabic",
        ]
    ].merge(
        prediction_df[
            [
                "source_id",
                "prediction",
                "raw_completion",
                "generation_error",
                "prompt_tokens",
                "completion_tokens",
                "harmony_message_count",
                "harmony_parse_error",
                "extraction_method",
            ]
        ],
        on="source_id",
        how="left",
        validate="one_to_one",
    )

    per_country_rows = []

    for country in sorted(
        scored_df["config"].unique()
    ):
        country_df = scored_df[
            scored_df["config"] == country
        ]

        predictions = (
            country_df["prediction"]
            .astype(str)
            .tolist()
        )

        references = (
            country_df[
                "reference_arabic"
            ]
            .astype(str)
            .tolist()
        )

        spbleu = (
            sacrebleu.corpus_bleu(
                predictions,
                [references],
                tokenize="flores200",
            ).score
        )

        chrfpp = (
            sacrebleu.corpus_chrf(
                predictions,
                [references],
                word_order=2,
            ).score
        )

        per_country_rows.append({
            "country": str(country),
            "turns": int(len(country_df)),
            "spBLEU": float(spbleu),
            "chrF++": float(chrfpp),
        })

    per_country_df = pd.DataFrame(
        per_country_rows
    )

    average_spbleu = float(
        per_country_df["spBLEU"].mean()
    )

    average_chrfpp = float(
        per_country_df["chrF++"].mean()
    )

    official_score_row = {
        "Experiment": EXPERIMENT_NAME,
        "Split": split_name,
        "Model": MODEL_ID,
        "Precision": PRECISION_MODE,
        "Reasoning effort": (
            REASONING_EFFORT
        ),
        "Decoding": "greedy",
        "Average spBLEU (primary)": (
            average_spbleu
        ),
        "Average chrF++": (
            average_chrfpp
        ),
    }

    for country_result in per_country_rows:
        country = country_result["country"]

        official_score_row[
            f"{country} spBLEU"
        ] = country_result["spBLEU"]

        official_score_row[
            f"{country} chrF++"
        ] = country_result["chrF++"]

    official_score_df = pd.DataFrame([
        official_score_row
    ])

    atomic_csv_write(
        scored_df,
        split_dir
        / "scored_turn_predictions.csv",
    )

    atomic_csv_write(
        per_country_df,
        split_dir
        / "per_country_official_metrics.csv",
    )

    atomic_csv_write(
        official_score_df,
        split_dir
        / "official_style_score_row.csv",
    )

    metrics = {
        "experiment": EXPERIMENT_NAME,
        "split": split_name,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "precision_mode": PRECISION_MODE,
        "model_torch_dtype": (
            MODEL_TORCH_DTYPE
        ),
        "reasoning_effort": (
            REASONING_EFFORT
        ),
        "decoding": "greedy",
        "fingerprint": fingerprint,
        "turns": int(len(scored_df)),
        "countries": int(
            scored_df["config"].nunique()
        ),
        "Average spBLEU (primary)": (
            average_spbleu
        ),
        "Average chrF++": average_chrfpp,
        "per_country": per_country_rows,
        "sacrebleu_version": (
            sacrebleu.__version__
        ),
        "spbleu_tokenizer": "flores200",
        "chrf_word_order": 2,
        "completed_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    atomic_json_write(
        metrics,
        split_dir
        / "official_metrics.json",
    )

    # --------------------------------------------------------
    # Create organizer-format predictions.jsonl
    # --------------------------------------------------------

    submission_records = []

    ordered_df = (
        scored_df
        .sort_values(
            [
                "config",
                "conversation_id",
                "turn_order",
            ]
        )
        .reset_index(drop=True)
    )

    grouped_conversations = (
        ordered_df.groupby(
            [
                "config",
                "conversation_id",
            ],
            sort=True,
        )
    )

    for (
        country,
        conversation_id,
    ), conversation_df in (
        grouped_conversations
    ):
        conversation_df = (
            conversation_df
            .sort_values("turn_order")
        )

        if conversation_df[
            "turn_order"
        ].duplicated().any():
            raise RuntimeError(
                "Duplicate turn inside "
                f"{country}/{conversation_id}."
            )

        turns = [
            {
                "turn_order": int(
                    row["turn_order"]
                ),
                "prediction": str(
                    row["prediction"]
                ),
            }
            for _, row in (
                conversation_df.iterrows()
            )
        ]

        submission_records.append({
            "conv_id": str(
                conversation_id
            ),
            "country": str(country),
            "turns": turns,
        })

    jsonl_path = (
        split_dir
        / "predictions.jsonl"
    )

    temporary_jsonl_path = (
        jsonl_path.with_name(
            jsonl_path.name + ".tmp"
        )
    )

    with open(
        temporary_jsonl_path,
        "w",
        encoding="utf-8",
    ) as file:
        for record in submission_records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    os.replace(
        temporary_jsonl_path,
        jsonl_path,
    )

    zip_path = (
        split_dir
        / "submission_predictions.zip"
    )

    temporary_zip_path = (
        zip_path.with_name(
            zip_path.name + ".tmp"
        )
    )

    with zipfile.ZipFile(
        temporary_zip_path,
        "w",
        compression=(
            zipfile.ZIP_DEFLATED
        ),
    ) as archive:
        archive.write(
            jsonl_path,
            arcname="predictions.jsonl",
        )

    os.replace(
        temporary_zip_path,
        zip_path,
    )

    # --------------------------------------------------------
    # Validate packaged ZIP by reading it back
    # --------------------------------------------------------

    readback_keys = set()

    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as archive:
        if archive.namelist() != [
            "predictions.jsonl"
        ]:
            raise RuntimeError(
                "The ZIP must contain only "
                "predictions.jsonl."
            )

        with archive.open(
            "predictions.jsonl"
        ) as file:
            for encoded_line in file:
                record = json.loads(
                    encoded_line.decode(
                        "utf-8"
                    )
                )

                for turn in record["turns"]:
                    key = (
                        str(record["country"]),
                        str(record["conv_id"]),
                        int(turn["turn_order"]),
                    )

                    if key in readback_keys:
                        raise RuntimeError(
                            "Duplicate packaged key: "
                            f"{key}"
                        )

                    readback_keys.add(key)

    expected_keys = set(zip(
        frame["config"].astype(str),
        frame[
            "conversation_id"
        ].astype(str),
        frame["turn_order"].astype(int),
    ))

    if readback_keys != expected_keys:
        raise RuntimeError(
            f"{split_name}: packaged key "
            "set mismatch."
        )

    print("\n" + "=" * 90)
    print(
        "GPT-OSS-20B TEACHER RESULT — "
        f"{split_name.upper()}"
    )
    print("=" * 90)

    print("Turns:", len(scored_df))
    print(
        "Countries:",
        scored_df["config"].nunique(),
    )

    print(
        "Average spBLEU (primary): "
        f"{average_spbleu:.4f}"
    )

    print(
        "Average chrF++: "
        f"{average_chrfpp:.4f}"
    )

    print("\nLeaderboard-style row:")
    display(official_score_df)

    print("\nPer-country metrics:")
    display(per_country_df)

    print("\nSaved folder:", split_dir)
    print("Predictions JSONL:", jsonl_path)
    print("Submission ZIP:", zip_path)

    return metrics


# ------------------------------------------------------------
# Main resumable benchmark runner
# ------------------------------------------------------------

def run_teacher_benchmark(
    frame,
    split_name,
):
    split_dir = (
        EXPERIMENT_DIR
        / split_name
    )

    split_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    predictions_path = (
        split_dir
        / "turn_predictions.csv"
    )

    fingerprint, payload = (
        experiment_fingerprint(
            frame,
            split_name,
        )
    )

    manifest_path = (
        split_dir
        / "experiment_manifest.json"
    )

    if manifest_path.exists():
        with open(
            manifest_path,
            "r",
            encoding="utf-8",
        ) as file:
            old_manifest = json.load(file)

        if (
            old_manifest.get("fingerprint")
            != fingerprint
        ):
            raise RuntimeError(
                "Experiment fingerprint mismatch in:\n"
                f"{split_dir}\n\n"
                "The folder contains predictions from a "
                "different model, precision, prompt, data, "
                "or decoding configuration. Use a new "
                "EXPERIMENT_NAME."
            )

    else:
        atomic_json_write(
            {
                **payload,
                "fingerprint": fingerprint,
                "created_at": time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
            },
            manifest_path,
        )

    expected_ids = set(
        frame["source_id"].astype(str)
    )

    original_order = dict(zip(
        frame["source_id"].astype(str),
        range(len(frame)),
    ))

    completed_rows = {}

    required_resume_columns = {
        "source_id",
        "prediction",
        "generation_error",
        "fingerprint",
    }

    # --------------------------------------------------------
    # Load only valid completed predictions
    # --------------------------------------------------------

    if predictions_path.exists():
        existing_df = pd.read_csv(
            predictions_path,
            dtype={
                "source_id": str,
            },
            keep_default_na=False,
        )

        missing_columns = (
            required_resume_columns
            - set(existing_df.columns)
        )

        if missing_columns:
            raise RuntimeError(
                "Existing prediction file has an "
                "incompatible format. Missing columns: "
                f"{sorted(missing_columns)}"
            )

        existing_df["source_id"] = (
            existing_df[
                "source_id"
            ].astype(str)
        )

        existing_df = (
            existing_df[
                existing_df["source_id"].isin(
                    expected_ids
                )
            ]
            .drop_duplicates(
                "source_id",
                keep="last",
            )
        )

        existing_fingerprints = set(
            existing_df["fingerprint"]
            .astype(str)
            .str.strip()
            .loc[
                lambda values:
                values.ne("")
            ]
        )

        if (
            existing_fingerprints
            and existing_fingerprints
            != {fingerprint}
        ):
            raise RuntimeError(
                "Existing predictions contain a "
                "different experiment fingerprint."
            )

        valid_mask = (
            existing_df["prediction"]
            .astype(str)
            .str.strip()
            .ne("")
        )

        valid_mask &= (
            existing_df[
                "generation_error"
            ]
            .astype(str)
            .str.strip()
            .eq("")
        )

        valid_mask &= (
            existing_df["fingerprint"]
            .astype(str)
            .str.strip()
            .eq(fingerprint)
        )

        valid_df = existing_df[
            valid_mask
        ]

        completed_rows = {
            str(row["source_id"]): row
            for row in valid_df.to_dict(
                "records"
            )
        }

        print(
            f"Resuming {split_name}: "
            f"{len(completed_rows)}/"
            f"{len(frame)} valid predictions."
        )

    else:
        print(
            f"Starting {split_name}: "
            f"0/{len(frame)} predictions."
        )

    # --------------------------------------------------------
    # Atomic progress saver
    # --------------------------------------------------------

    def save_progress(rows_by_id):
        saved_df = pd.DataFrame(
            list(rows_by_id.values())
        )

        if len(saved_df):
            saved_df["source_id"] = (
                saved_df[
                    "source_id"
                ].astype(str)
            )

            saved_df["_original_order"] = (
                saved_df["source_id"]
                .map(original_order)
            )

            saved_df = (
                saved_df
                .sort_values(
                    "_original_order"
                )
                .drop(
                    columns=[
                        "_original_order"
                    ]
                )
                .reset_index(drop=True)
            )

        atomic_csv_write(
            saved_df,
            predictions_path,
        )

        return saved_df

    generated_since_save = 0

    # --------------------------------------------------------
    # Generate missing or previously failed rows
    # --------------------------------------------------------

    try:
        for _, row in tqdm(
            frame.iterrows(),
            total=len(frame),
            desc=f"GPT-OSS {split_name}",
        ):
            row_dict = row.to_dict()

            source_id = str(
                row_dict["source_id"]
            )

            if source_id in completed_rows:
                continue

            try:
                generation_result = (
                    generate_one(row_dict)
                )

            except Exception as error:
                generation_result = {
                    "raw_completion": "",
                    "prediction": "",
                    "generation_error": (
                        f"{type(error).__name__}: "
                        f"{str(error)[:1000]}"
                    ),
                    "prompt_tokens": 0,
                    "completion_tokens": 0,
                    "harmony_message_count": 0,
                    "harmony_parse_error": "",
                    "extraction_method": (
                        "generation_exception"
                    ),
                }

            completed_rows[source_id] = {
                "source_id": source_id,
                "config": row_dict["config"],
                "country": row_dict["country"],
                "conversation_id": (
                    row_dict[
                        "conversation_id"
                    ]
                ),
                "turn_order": int(
                    row_dict["turn_order"]
                ),
                "source_text": (
                    row_dict["source_text"]
                ),

                **generation_result,

                "model_id": MODEL_ID,
                "model_revision": (
                    MODEL_REVISION
                ),
                "precision_mode": (
                    PRECISION_MODE
                ),
                "reasoning_effort": (
                    REASONING_EFFORT
                ),
                "decode_tag": DECODE_TAG,
                "prompt_version": (
                    PROMPT_VERSION
                ),
                "fingerprint": fingerprint,
                "generated_at": time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
            }

            generated_since_save += 1

            if (
                generated_since_save
                >= SAVE_EVERY
            ):
                saved_df = save_progress(
                    completed_rows
                )

                print(
                    f"Saved {len(saved_df)}/"
                    f"{len(frame)} rows."
                )

                generated_since_save = 0

    finally:
        # KeyboardInterrupt is intentionally not caught.
        # Successfully completed rows are saved first.
        if completed_rows:
            save_progress(completed_rows)

    # --------------------------------------------------------
    # Final completeness validation
    # --------------------------------------------------------

    final_df = pd.read_csv(
        predictions_path,
        dtype={
            "source_id": str,
        },
        keep_default_na=False,
    )

    final_df["source_id"] = (
        final_df["source_id"].astype(str)
    )

    final_df["prediction"] = (
        final_df["prediction"]
        .astype(str)
        .str.strip()
    )

    final_df["generation_error"] = (
        final_df["generation_error"]
        .astype(str)
        .str.strip()
    )

    final_ids = set(
        final_df["source_id"]
    )

    missing_ids = (
        expected_ids - final_ids
    )

    unexpected_ids = (
        final_ids - expected_ids
    )

    duplicate_count = int(
        final_df[
            "source_id"
        ].duplicated().sum()
    )

    empty_count = int(
        final_df[
            "prediction"
        ].eq("").sum()
    )

    error_count = int(
        final_df[
            "generation_error"
        ].ne("").sum()
    )

    wrong_fingerprint_count = int(
        final_df["fingerprint"]
        .astype(str)
        .ne(fingerprint)
        .sum()
    )

    print(
        f"\nFinal {split_name} validation"
    )
    print("-" * 72)
    print(
        "Rows:",
        len(final_df),
        "/",
        len(frame),
    )
    print("Missing:", len(missing_ids))
    print("Unexpected:", len(unexpected_ids))
    print("Duplicates:", duplicate_count)
    print("Empty:", empty_count)
    print("Errors:", error_count)
    print(
        "Wrong fingerprints:",
        wrong_fingerprint_count,
    )

    incomplete = any([
        bool(missing_ids),
        bool(unexpected_ids),
        duplicate_count > 0,
        empty_count > 0,
        error_count > 0,
        wrong_fingerprint_count > 0,
        len(final_df) != len(frame),
    ])

    if incomplete:
        problem_mask = (
            final_df["prediction"].eq("")
            | final_df[
                "generation_error"
            ].ne("")
        )

        if problem_mask.any():
            display(
                final_df.loc[
                    problem_mask,
                    [
                        "source_id",
                        "source_text",
                        "prediction",
                        "generation_error",
                        "harmony_parse_error",
                        "extraction_method",
                    ],
                ].head(20)
            )

        raise RuntimeError(
            f"{split_name} is incomplete. "
            "Rerun this same cell to retain valid "
            "predictions and retry only failed rows."
        )

    return score_and_package(
        frame=frame,
        prediction_df=final_df,
        split_name=split_name,
        split_dir=split_dir,
        fingerprint=fingerprint,
    )


print(
    "✅ Native-MXFP4 teacher benchmark "
    "engine is ready."
)

✅ Native-MXFP4 teacher benchmark engine is ready.


In [25]:
# ============================================================
# Recovery patch — Retry only invalid GPT-OSS generations
#
# Run this cell AFTER Cell 6 and BEFORE rerunning Cell 8.
#
# Existing valid predictions remain untouched.
# Failed predictions receive:
#   1. the original deterministic attempt
#   2. a stricter deterministic recovery attempt
#   3. a stronger deterministic recovery attempt if needed
# ============================================================

import copy
import re

import torch


RECOVERY_POLICY_VERSION = (
    "deterministic_repetition_recovery_v1"
)


# ------------------------------------------------------------
# Preserve the original generate_one function
# ------------------------------------------------------------

if (
    "_original_generate_one_before_recovery"
    not in globals()
):
    _original_generate_one_before_recovery = (
        generate_one
    )


# ------------------------------------------------------------
# Detect pathological repetitive outputs
# ------------------------------------------------------------

def is_degenerate_translation(text):
    text = clean_translation(text)

    if not text:
        return True

    # Repeated characters or Arabic tatweel.
    if re.search(r"(.)\1{7,}", text):
        return True

    words = re.findall(
        r"\S+",
        text.lower(),
    )

    if len(words) < 8:
        return False

    # One word dominates the output.
    most_common_count = max(
        words.count(word)
        for word in set(words)
    )

    if (
        most_common_count
        / len(words)
        >= 0.35
    ):
        return True

    # Excessive repeated 3-word sequences.
    trigrams = [
        tuple(words[index:index + 3])
        for index in range(
            len(words) - 2
        )
    ]

    if trigrams:
        repetition_fraction = (
            1.0
            - (
                len(set(trigrams))
                / len(trigrams)
            )
        )

        if repetition_fraction >= 0.45:
            return True

    return False


# ------------------------------------------------------------
# One deterministic recovery attempt
# ------------------------------------------------------------

def generate_recovery_attempt(
    row,
    *,
    max_new_tokens,
    repetition_penalty,
    no_repeat_ngram_size,
    attempt_name,
):
    messages = copy.deepcopy(
        build_messages(row)
    )

    strict_instruction = """
Critical output constraints:

- Produce one concise Arabic translation.
- Do not repeat words, phrases, letters, or punctuation.
- Do not reproduce the English source.
- Do not add dialect labels or quotation marks.
- Finish immediately after the translation.
""".strip()

    messages[0]["content"] = (
        messages[0]["content"]
        + "\n\n"
        + strict_instruction
    )

    messages[1]["content"] = (
        messages[1]["content"]
        + "\n\n"
        + (
            "Translate once and stop. "
            "Do not repeat any part of the answer."
        )
    )

    encoded = (
        teacher_tokenizer
        .apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
            reasoning_effort=(
                REASONING_EFFORT
            ),
        )
    )

    encoded = {
        key: value.to(INPUT_DEVICE)
        for key, value in encoded.items()
    }

    prompt_tokens = int(
        encoded["input_ids"].shape[-1]
    )

    with torch.inference_mode():
        generated = teacher_model.generate(
            **encoded,

            max_new_tokens=max_new_tokens,

            # Still fully deterministic.
            do_sample=False,
            num_beams=1,

            # Prevent pathological loops.
            repetition_penalty=(
                repetition_penalty
            ),
            no_repeat_ngram_size=(
                no_repeat_ngram_size
            ),

            eos_token_id=(
                HARMONY_STOP_TOKEN_IDS
            ),
            pad_token_id=(
                teacher_tokenizer
                .pad_token_id
            ),
            use_cache=True,
        )

    completion_ids = (
        generated[
            0,
            prompt_tokens:,
        ]
        .detach()
        .cpu()
        .tolist()
    )

    raw_completion = (
        teacher_tokenizer.decode(
            completion_ids,
            skip_special_tokens=False,
        )
    )

    extraction = extract_final_channel(
        completion_ids=completion_ids,
        raw_completion=raw_completion,
    )

    prediction = extraction["prediction"]

    reached_token_limit = (
        len(completion_ids)
        >= max_new_tokens
        and (
            not completion_ids
            or completion_ids[-1]
            not in HARMONY_STOP_TOKEN_ID_SET
        )
    )

    generation_error = ""

    if reached_token_limit:
        generation_error = (
            "recovery_max_new_tokens_reached"
        )

    elif not prediction:
        generation_error = (
            "recovery_missing_harmony_final"
        )

    elif arabic_ratio(prediction) < 0.10:
        generation_error = (
            "recovery_low_arabic_ratio"
        )

    elif is_degenerate_translation(
        prediction
    ):
        generation_error = (
            "recovery_degenerate_repetition"
        )

    return {
        "raw_completion": raw_completion,
        "prediction": prediction,
        "generation_error": (
            generation_error
        ),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": len(
            completion_ids
        ),
        "harmony_message_count": len(
            extraction["parsed_messages"]
        ),
        "harmony_parse_error": (
            extraction[
                "harmony_parse_error"
            ]
        ),
        "extraction_method": (
            extraction[
                "extraction_method"
            ]
        ),
        "recovery_policy": (
            RECOVERY_POLICY_VERSION
        ),
        "recovery_attempt": attempt_name,
    }


# ------------------------------------------------------------
# Replace generate_one with primary + recovery behavior
# ------------------------------------------------------------

def generate_one(row):
    # First try the original experiment configuration.
    primary_result = (
        _original_generate_one_before_recovery(
            row
        )
    )

    primary_error = safe_text(
        primary_result.get(
            "generation_error",
            "",
        )
    )

    primary_prediction = safe_text(
        primary_result.get(
            "prediction",
            "",
        )
    )

    primary_degenerate = (
        bool(primary_prediction)
        and is_degenerate_translation(
            primary_prediction
        )
    )

    if (
        not primary_error
        and not primary_degenerate
    ):
        primary_result[
            "recovery_policy"
        ] = ""

        primary_result[
            "recovery_attempt"
        ] = "primary"

        primary_result[
            "initial_generation_error"
        ] = ""

        return primary_result

    initial_error = (
        primary_error
        or "primary_degenerate_repetition"
    )

    recovery_strategies = [
        {
            "max_new_tokens": max(
                MAX_NEW_TOKENS,
                384,
            ),
            "repetition_penalty": 1.05,
            "no_repeat_ngram_size": 6,
            "attempt_name": (
                "deterministic_recovery_1"
            ),
        },
        {
            "max_new_tokens": max(
                MAX_NEW_TOKENS,
                512,
            ),
            "repetition_penalty": 1.12,
            "no_repeat_ngram_size": 4,
            "attempt_name": (
                "deterministic_recovery_2"
            ),
        },
    ]

    last_result = primary_result

    for strategy in recovery_strategies:
        recovery_result = (
            generate_recovery_attempt(
                row,
                **strategy,
            )
        )

        recovery_result[
            "initial_generation_error"
        ] = initial_error

        last_result = recovery_result

        if not recovery_result[
            "generation_error"
        ]:
            return recovery_result

    return last_result


print(
    "✅ Deterministic recovery policy installed."
)

print(
    "Rerun Cell 8. Existing valid rows will "
    "be skipped and only invalid rows retried."
)

✅ Deterministic recovery policy installed.
Rerun Cell 8. Existing valid rows will be skipped and only invalid rows retried.


#### Smoke Test

In [9]:
# ============================================================
# Cell 7 — Mandatory native-MXFP4 smoke test
#
# Do not start complete inference unless all examples:
#   - contain a non-empty Harmony final message
#   - produce Arabic output
#   - finish without generation errors
# ============================================================

smoke_df = (
    dev_df
    .groupby(
        "config",
        sort=True,
    )
    .head(1)
    .head(3)
)

if smoke_df.empty:
    raise RuntimeError(
        "Smoke-test dataframe is empty."
    )

for _, smoke_row in smoke_df.iterrows():
    smoke_result = generate_one(
        smoke_row.to_dict()
    )

    print("\n" + "=" * 80)
    print(
        "Country:",
        smoke_row["config"],
    )
    print(
        "English:",
        smoke_row["source_text"],
    )
    print(
        "Reference:",
        smoke_row["reference_arabic"],
    )
    print(
        "Prediction:",
        smoke_result["prediction"],
    )
    print(
        "Arabic ratio:",
        f"{arabic_ratio(smoke_result['prediction']):.3f}",
    )
    print(
        "Extraction:",
        smoke_result[
            "extraction_method"
        ],
    )
    print(
        "Harmony messages:",
        smoke_result[
            "harmony_message_count"
        ],
    )
    print(
        "Prompt tokens:",
        smoke_result["prompt_tokens"],
    )
    print(
        "Completion tokens:",
        smoke_result[
            "completion_tokens"
        ],
    )
    print(
        "Error:",
        smoke_result[
            "generation_error"
        ],
    )

    if smoke_result["generation_error"]:
        print("\nRaw Harmony completion:")
        print(
            smoke_result[
                "raw_completion"
            ]
        )

        if smoke_result[
            "harmony_parse_error"
        ]:
            print(
                "\nHarmony parse error:"
            )
            print(
                smoke_result[
                    "harmony_parse_error"
                ]
            )

        raise RuntimeError(
            "Smoke test failed. Do not start "
            "complete inference."
        )

print(
    "\n✅ Native-MXFP4 smoke test passed."
)


Country: EG
English: Good news, the ministry has finally approved the budget for the barrage repairs.
Reference: أخبار حلوة، الوزارة أخيرا وافقت على الميزانية بتاعة تصليح القناطر.
Prediction: أخبار حلوة، الوزارة أخيرًا وافقت على ميزانية إصلاح السد.
Arabic ratio: 0.979
Extraction: openai_harmony_parser
Harmony messages: 2
Prompt tokens: 267
Completion tokens: 35
Error: 

Country: JO
English: My friend, we have an issue. The zucchini boxes have the good ones on top, but the bottom is full of small and bruised pieces.
Reference: صديقي في عنا مشكلة، بوكس الكوسا اللي من فوق مليحات بس اللي تحت مليانة حبات صغيرة ومضروبه
Prediction: صديقي، عندنا مشكلة. علب الكوسا فيها الجُزء الجيد فوق، لكن الجزء السفلي مليان قطع صغيرة ومضغوطة.
Arabic ratio: 0.975
Extraction: openai_harmony_parser
Harmony messages: 2
Prompt tokens: 276
Completion tokens: 51
Error: 

Country: LB
English: Good morning. I'm calling to place my main order for the upcoming winter season.
Reference: صباح الخير. عم دقلك لحتى وصي على 

### **Complete Training Set Inference**

In [26]:
# ============================================================
# Cell 8 — Complete TRAIN inference and scoring
#

# Safe to rerun:
#   - valid completed rows are skipped
#   - failed rows are retried
#   - progress is saved every SAVE_EVERY rows
# ============================================================

train_metrics = run_teacher_benchmark(
    frame=train_df,
    split_name="train",
)

Resuming train: 66396/66480 valid predictions.


GPT-OSS train:   0%|          | 0/66480 [00:00<?, ?it/s]


Final train validation
------------------------------------------------------------------------
Rows: 66480 / 66480
Missing: 0
Unexpected: 0
Duplicates: 0
Empty: 0
Errors: 0
Wrong fingerprints: 0

GPT-OSS-20B TEACHER RESULT — TRAIN
Turns: 66480
Countries: 11
Average spBLEU (primary): 19.3712
Average chrF++: 36.0222

Leaderboard-style row:


,Experiment,Split,Model,Precision,Reasoning effort,Decoding,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,gpt_oss_20b_dequant_bf16_greedy_noshot_context...,train,openai/gpt-oss-20b,native_mxfp4,low,greedy,19.371209,36.022191,22.803506,38.259475,...,21.45452,37.967113,26.381871,43.21295,23.36325,40.862274,19.060262,33.531112,17.296381,35.026916



Per-country metrics:


,country,turns,spBLEU,chrF++
0,EG,3108,22.803506,38.259475
1,JO,5501,22.711048,40.329391
2,LB,8906,18.130709,35.037958
3,MA,2573,15.045249,31.209531
4,MR,5515,8.932628,25.597858
5,OM,6280,17.903880,35.209522
6,PS,14933,21.454520,37.967113
7,SA,8470,26.381871,43.212950
8,SY,6071,23.363250,40.862274
9,TN,2034,19.060262,33.531112



Saved folder: /home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev/train
Predictions JSONL: /home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev/train/predictions.jsonl
Submission ZIP: /home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev/train/submission_predictions.zip


### **Complete Dev Set Inference**

In [28]:
# ============================================================
# Cell 9 — Complete official DEV inference and scoring
#
# Safe to rerun:
#   - valid completed rows are skipped
#   - failed rows are retried
#   - predictions.jsonl and submission ZIP are generated
# ============================================================

dev_metrics = run_teacher_benchmark(
    frame=dev_df,
    split_name="dev",
)

Resuming dev: 9270/12250 valid predictions.


GPT-OSS dev:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved 9470/12250 rows.
Saved 9570/12250 rows.
Saved 9670/12250 rows.
Saved 9770/12250 rows.
Saved 9870/12250 rows.
Saved 9970/12250 rows.
Saved 10070/12250 rows.
Saved 10170/12250 rows.
Saved 10270/12250 rows.
Saved 10370/12250 rows.
Saved 10470/12250 rows.
Saved 10570/12250 rows.
Saved 10670/12250 rows.
Saved 10770/12250 rows.
Saved 10870/12250 rows.
Saved 10970/12250 rows.


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Saved 11470/12250 rows.
Saved 11570/12250 rows.
Saved 11670/12250 rows.
Saved 11770/12250 rows.
Saved 11870/12250 rows.
Saved 11970/12250 rows.
Saved 12070/12250 rows.
Saved 12170/12250 rows.

Final dev validation
------------------------------------------------------------------------
Rows: 12250 / 12250
Missing: 0
Unexpected: 0
Duplicates: 0
Empty: 0
Errors: 0
Wrong fingerprints: 0

GPT-OSS-20B TEACHER RESULT — DEV
Turns: 12250
Countries: 11
Average spBLEU (primary): 19.2811
Average chrF++: 35.9952

Leaderboard-style row:


,Experiment,Split,Model,Precision,Reasoning effort,Decoding,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,gpt_oss_20b_dequant_bf16_greedy_noshot_context...,dev,openai/gpt-oss-20b,native_mxfp4,low,greedy,19.281131,35.995173,22.770787,38.617993,...,21.50762,37.820315,23.273619,40.601841,24.153397,41.588894,18.828197,33.94957,18.839553,36.314841



Per-country metrics:


,country,turns,spBLEU,chrF++
0,EG,1113,22.770787,38.617993
1,JO,1113,22.397572,39.860152
2,LB,1118,18.089374,34.938120
3,MA,1110,14.767452,30.684441
4,MR,1114,8.878736,25.623607
5,OM,1109,18.586137,35.947125
6,PS,1110,21.507620,37.820315
7,SA,1110,23.273619,40.601841
8,SY,1119,24.153397,41.588894
9,TN,1116,18.828197,33.949570



Saved folder: /home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev/dev
Predictions JSONL: /home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev/dev/predictions.jsonl
Submission ZIP: /home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev/dev/submission_predictions.zip


### **Final Train vs Dev comparison**

In [29]:
# ============================================================
# Cell 10 — Final TRAIN versus DEV comparison
# ============================================================

# Allow this cell to work after a kernel restart if both
# inference runs have already completed.

if "train_metrics" not in globals():
    train_metrics_path = (
        EXPERIMENT_DIR
        / "train"
        / "official_metrics.json"
    )

    if not train_metrics_path.exists():
        raise RuntimeError(
            "TRAIN metrics are unavailable. "
            "Run Cell 8 first."
        )

    with open(
        train_metrics_path,
        "r",
        encoding="utf-8",
    ) as file:
        train_metrics = json.load(file)


if "dev_metrics" not in globals():
    dev_metrics_path = (
        EXPERIMENT_DIR
        / "dev"
        / "official_metrics.json"
    )

    if not dev_metrics_path.exists():
        raise RuntimeError(
            "DEV metrics are unavailable. "
            "Run Cell 9 first."
        )

    with open(
        dev_metrics_path,
        "r",
        encoding="utf-8",
    ) as file:
        dev_metrics = json.load(file)


summary_df = pd.DataFrame([
    {
        "split": "train",
        "turns": train_metrics["turns"],
        "countries": (
            train_metrics["countries"]
        ),
        "precision": (
            train_metrics[
                "precision_mode"
            ]
        ),
        "reasoning_effort": (
            train_metrics[
                "reasoning_effort"
            ]
        ),
        "decoding": (
            train_metrics["decoding"]
        ),
        "Average spBLEU (primary)": (
            train_metrics[
                "Average spBLEU (primary)"
            ]
        ),
        "Average chrF++": (
            train_metrics[
                "Average chrF++"
            ]
        ),
    },
    {
        "split": "dev",
        "turns": dev_metrics["turns"],
        "countries": (
            dev_metrics["countries"]
        ),
        "precision": (
            dev_metrics[
                "precision_mode"
            ]
        ),
        "reasoning_effort": (
            dev_metrics[
                "reasoning_effort"
            ]
        ),
        "decoding": (
            dev_metrics["decoding"]
        ),
        "Average spBLEU (primary)": (
            dev_metrics[
                "Average spBLEU (primary)"
            ]
        ),
        "Average chrF++": (
            dev_metrics[
                "Average chrF++"
            ]
        ),
    },
])

atomic_csv_write(
    summary_df,
    EXPERIMENT_DIR
    / "train_dev_summary.csv",
)


train_country_df = pd.read_csv(
    EXPERIMENT_DIR
    / "train"
    / "per_country_official_metrics.csv"
).rename(
    columns={
        "turns": "train_turns",
        "spBLEU": "train_spBLEU",
        "chrF++": "train_chrF++",
    }
)


dev_country_df = pd.read_csv(
    EXPERIMENT_DIR
    / "dev"
    / "per_country_official_metrics.csv"
).rename(
    columns={
        "turns": "dev_turns",
        "spBLEU": "dev_spBLEU",
        "chrF++": "dev_chrF++",
    }
)


country_comparison_df = (
    train_country_df.merge(
        dev_country_df,
        on="country",
        how="outer",
        validate="one_to_one",
    )
    .sort_values("country")
    .reset_index(drop=True)
)


country_comparison_df[
    "spBLEU_gap_dev_minus_train"
] = (
    country_comparison_df[
        "dev_spBLEU"
    ]
    - country_comparison_df[
        "train_spBLEU"
    ]
)


country_comparison_df[
    "chrF++_gap_dev_minus_train"
] = (
    country_comparison_df[
        "dev_chrF++"
    ]
    - country_comparison_df[
        "train_chrF++"
    ]
)


atomic_csv_write(
    country_comparison_df,
    EXPERIMENT_DIR
    / "train_dev_per_country_comparison.csv",
)


print("=" * 90)
print("GPT-OSS-20B NATIVE MXFP4 FINAL SUMMARY")
print("=" * 90)

print("\nOverall TRAIN versus DEV:")
display(summary_df)

print("\nPer-country TRAIN versus DEV:")
display(country_comparison_df)

print("\nExperiment folder:")
print(EXPERIMENT_DIR)

print("\nOfficial DEV predictions:")
print(
    EXPERIMENT_DIR
    / "dev"
    / "predictions.jsonl"
)

print("\nOfficial DEV-format ZIP:")
print(
    EXPERIMENT_DIR
    / "dev"
    / "submission_predictions.zip"
)

print(
    "\n✅ Native-MXFP4 TRAIN and DEV "
    "benchmark workflow completed."
)

GPT-OSS-20B NATIVE MXFP4 FINAL SUMMARY

Overall TRAIN versus DEV:


,split,turns,countries,precision,reasoning_effort,decoding,Average spBLEU (primary),Average chrF++
0,train,66480,11,native_mxfp4,low,greedy,19.371209,36.022191
1,dev,12250,11,native_mxfp4,low,greedy,19.281131,35.995173



Per-country TRAIN versus DEV:


,country,train_turns,train_spBLEU,train_chrF++,dev_turns,dev_spBLEU,dev_chrF++,spBLEU_gap_dev_minus_train,chrF++_gap_dev_minus_train
0,EG,3108,22.803506,38.259475,1113,22.770787,38.617993,-0.032719,0.358518
1,JO,5501,22.711048,40.329391,1113,22.397572,39.860152,-0.313475,-0.469239
2,LB,8906,18.130709,35.037958,1118,18.089374,34.938120,-0.041335,-0.099837
3,MA,2573,15.045249,31.209531,1110,14.767452,30.684441,-0.277797,-0.525090
4,MR,5515,8.932628,25.597858,1114,8.878736,25.623607,-0.053892,0.025749
5,OM,6280,17.903880,35.209522,1109,18.586137,35.947125,0.682257,0.737604
6,PS,14933,21.454520,37.967113,1110,21.507620,37.820315,0.053100,-0.146798
7,SA,8470,26.381871,43.212950,1110,23.273619,40.601841,-3.108252,-2.611109
8,SY,6071,23.363250,40.862274,1119,24.153397,41.588894,0.790147,0.726620
9,TN,2034,19.060262,33.531112,1116,18.828197,33.949570,-0.232066,0.418458



Experiment folder:
/home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev

Official DEV predictions:
/home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev/dev/predictions.jsonl

Official DEV-format ZIP:
/home/mabdallah/alexandriax_mt_14d/teacher_benchmarks/gpt_oss_20b_dequant_bf16_greedy_noshot_context3_train_dev/dev/submission_predictions.zip

✅ Native-MXFP4 TRAIN and DEV benchmark workflow completed.
